# MT Benchmark — Turkish → English (Helsinki)

Evaluates `Helsinki-NLP/opus-mt-tr-en` on Turkish→English translation. Run alongside [`turkish_mbart_benchmark.ipynb`](turkish_mbart_benchmark.ipynb) to compare.

**Dataset:** MaCoCu Turkish–English (Option A) or Tatoeba (Option B, no Drive required)  
**Metrics:** METEOR, BERTScore, XLMrScore  
**Results:** See [`README.md`](README.md)

In [ ]:
# Uncomment when running on Colab
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [ ]:
%%capture
!pip install transformers sentencepiece bert_score nltk accelerate datasets
import nltk
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import MarianMTModel, MarianTokenizer, AutoTokenizer
from bert_score import score as bert_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
SAMPLE_SIZE = 10_000
BATCH_SIZE  = 16      # Helsinki is faster — can use larger batches
MAX_LENGTH  = 512
# ──────────────────────────────────────────────────────────────────────────────

## 1. Load dataset

Use the same dataset and sample as the MBART notebook for a like-for-like comparison.

In [ ]:
# ── Option A: Tatoeba (no Drive) ─────────────────────────────────────────────
from datasets import load_dataset

raw = load_dataset('Helsinki-NLP/tatoeba_mt', 'tur-eng', split='test', trust_remote_code=True)
df = pd.DataFrame({
    'source': [r['sourceString'] for r in raw],
    'target': [r['targetString'] for r in raw],
})
df = df.dropna().sample(n=min(SAMPLE_SIZE, len(df)), random_state=1).reset_index(drop=True)
print(f'{len(df):,} sentence pairs')

In [ ]:
# ── Option B: MaCoCu from Drive (uncomment to use) ────────────────────────────
# FILE_PATH = '/content/drive/MyDrive/YOUR_PROJECT/data/MaCoCu-tr-en.sent.txt'
#
# raw = pd.read_csv(FILE_PATH, sep='\t', on_bad_lines='skip')
# raw = raw[raw['translation_direction'] == 'first-orig-second-ht']
# raw['bleualign_score']    = raw['bleualign_score'].astype(float)
# raw['bicleaner_ai_score'] = raw['bicleaner_ai_score'].astype(float)
# raw = raw[raw['bicleaner_ai_score'] > 0.9]
# raw = raw[raw['bleualign_score'] > 0.4]
# df = raw[['src_text', 'trg_text']].rename(columns={'src_text': 'source', 'trg_text': 'target'})
# df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=1).reset_index(drop=True)

## 2. Translate — Helsinki opus-mt-tr-en

Marian models do not require a `forced_bos_token_id` — the target language is fixed at model load time.

In [ ]:
model_name = 'Helsinki-NLP/opus-mt-tr-en'
tokenizer  = MarianTokenizer.from_pretrained(model_name)
model      = MarianMTModel.from_pretrained(model_name)

model.to(DEVICE).eval()
print(f'Model loaded on {DEVICE}')

In [ ]:
translations = []
for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch = list(df['source'].iloc[i:i + BATCH_SIZE])
    enc = tokenizer(
        batch, return_tensors='pt',
        padding=True, truncation=True, max_length=MAX_LENGTH,
    ).to(DEVICE)
    with torch.no_grad():
        tokens = model.generate(**enc)
    translations.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True))

df['translation'] = translations
print('Done.')

## 3. METEOR

In [ ]:
meteor_fn = nltk.translate.meteor_score.meteor_score
scores = []
for ref, hyp in tqdm(zip(df['target'], df['translation']), total=len(df)):
    scores.append(meteor_fn([ref.split()], hyp.split()))
df['meteor'] = scores
print(f'Average METEOR: {np.mean(scores):.4f}')

## 4. BERTScore

In [ ]:
tok   = AutoTokenizer.from_pretrained('roberta-large')
refs  = [' '.join(tok.tokenize(r)) for r in df['target']]
cands = [' '.join(tok.tokenize(h)) for h in df['translation']]

_, _, F1 = bert_score(cands, refs, lang='en', model_type='roberta-large', verbose=True)
df['bertscore'] = F1.numpy()
print(f'Average BERTScore F1: {F1.mean():.4f}')

## 5. XLMrScore

In [ ]:
xlm_tok = AutoTokenizer.from_pretrained('xlm-roberta-base')
src_tok = [' '.join(xlm_tok.tokenize(s)) for s in df['source']]
hyp_tok = [' '.join(xlm_tok.tokenize(h)) for h in df['translation']]

_, _, F1_xlm = bert_score(hyp_tok, src_tok, model_type='xlm-roberta-base', verbose=True)
df['xlmrscore'] = F1_xlm.numpy()
print(f'Average XLMrScore F1: {F1_xlm.mean():.4f}')

## 6. Results

In [ ]:
print(df[['meteor', 'bertscore', 'xlmrscore']].describe().round(4))

In [ ]:
print('Reference results from original experiment (10K MaCoCu, GPU A100, batch=16):')
print('  METEOR:    0.500')
print('  BERTScore: 0.929')
print('  XLMrScore: 0.852')
print('  GPU time:  20 min')
print()
print('vs MBART on same data: METEOR=0.534, BERTScore=0.929, XLMrScore=0.852, GPU=41min')
print('BERTScore/XLMrScore gap: negligible. METEOR gap: 0.034 (MBART better).')

## 7. Error analysis

In [ ]:
low = df[df['bertscore'] < 0.9].sample(n=min(20, len(df[df['bertscore'] < 0.9])), random_state=1)

for _, row in low.iterrows():
    print(f'SOURCE:      {row["source"]}')
    print(f'GOLD:        {row["target"]}')
    print(f'TRANSLATION: {row["translation"]}')
    print(f'METEOR={row["meteor"]:.3f}  BERTScore={row["bertscore"]:.3f}  XLMrScore={row["xlmrscore"]:.3f}')
    print()